# V99 - Top-5 set filter then order

Pre-registered plan: ranking conflates set selection with ordering, so split them. Per-module logistic classifiers predict top-5 membership from word-only features, then Ridge orders within the predicted set first and the rest after. Membership labels come from graded ranks with ties broken stably. Same grouped folds under a fresh seed, same locks. If the set is easier than the order, this wins.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold

SEED = 2525
N_SPLITS = 5
GATE_FLOOR = 0.002
rng = np.random.RandomState(SEED)

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v99")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

DATA: /home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender/data


## Membership OOF, lexicographic order, gate

Members sort by Ridge above non-members. The baseline is plain Ridge order on identical folds.

In [2]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in list(train["user_id"])]
W = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs).toarray()
MEM = np.zeros_like(Y)
for u in range(n_train):
    MEM[u, np.argsort(-Y[u], kind="stable")[:5]] = 1.0

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

oof_r, oof_p, fid = np.zeros_like(Y), np.zeros_like(Y), np.zeros(n_train, int)
skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
for fi, (tr, va) in enumerate(skf.split(W, strata, fold_groups)):
    for j in range(len(MODS)):
        m = Ridge(alpha=1.0)
        m.fit(W[tr], Y[tr, j])
        oof_r[va, j] = m.predict(W[va])
        c = LogisticRegression(max_iter=500)
        c.fit(W[tr], MEM[tr, j])
        oof_p[va, j] = c.predict_proba(W[va])[:, 1]
    fid[va] = fi + 1
final = np.zeros_like(Y)
for u in range(n_train):
    order = np.lexsort((-oof_r[u], -oof_p[u]))
    final[u, order] = np.linspace(1.0, 0.0, len(MODS))
base = ndcg5(Y, oof_r)
d = ndcg5(Y, final) - base
lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
wf = min(ndcg5(Y[fid == k], final[fid == k]).mean() - base[fid == k].mean() for k in range(1, N_SPLITS + 1))
print(f"rank-only={base.mean():.5f} set+order={ndcg5(Y, final).mean():.5f} delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f}")
print("ADOPT set filter" if (lo > GATE_FLOOR and wf >= 0) else "KEEP pure ranking; set prediction adds nothing")

rank-only=0.62273 set+order=0.59567 delta=-0.02706 CI95=[-0.03059,-0.02340] worst-fold=-0.04621
KEEP pure ranking; set prediction adds nothing
